# 03장 보안 실습 — IOC 검색과 입력 경계


## Goal

문자열·정규식·필드 비교와 검색 실패를 구분합니다.

[교안과 분석 질문](../../03-bash-basics/03-10-ioc-search.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-03-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'ioc.log': '2026-09-10T09:01:00+09:00 src=192.0.2.10 action=failed\n2026-09-10T09:02:00+09:00 src=192.0.2.100 action=accepted\n2026-09-10T09:03:00+09:00 note=indicator.example\n2026-09-10T09:04:00+09:00 note=indicatorXexample\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: literal=1 regex=2; substring=2 exact_field=1; no_match_status=1

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 제공 도구 준비

아래 구현을 읽고 입력 검증·실패 처리·출력 경계를 표시합니다. 실행 중 다운로드하지 않으며 실습 폴더에만 저장합니다.


In [ ]:
%%bash
set -euo pipefail
cat > "$COURSE_TOOLS/ioc-search.sh" <<'COURSE_TOOL'
#!/usr/bin/env bash
# Literal line search only; does not connect to indicators or execute input.
set -u
usage() { printf 'usage: bash ioc-search.sh -f FILE -i LITERAL\n' >&2; }
input_file='' indicator='' seen_file=0 seen_indicator=0
while getopts ':f:i:' option; do
    case $option in
        f) (( seen_file == 0 )) || { usage; exit 2; }; input_file=$OPTARG; seen_file=1 ;;
        i) (( seen_indicator == 0 )) || { usage; exit 2; }; indicator=$OPTARG; seen_indicator=1 ;;
        *) usage; exit 2 ;;
    esac
done
shift "$((OPTIND - 1))"
if (( $# != 0 || seen_file != 1 || seen_indicator != 1 )) || [[ -z $input_file || -z $indicator || $indicator == *$'\n'* ]]; then
    usage; exit 2
fi
if [[ ! -f $input_file || ! -r $input_file ]]; then
    printf 'input is not a readable regular file\n' >&2; exit 2
fi
# grep: 0 match, 1 no match, 2 error. Read errors can follow partial stdout.
grep -nF -- "$indicator" "$input_file"
status=$?
if (( status > 1 )); then
    printf 'search failed; discard partial results\n' >&2
    exit 2
fi
exit "$status"
COURSE_TOOL


### 1. 문자열과 정규식 비교


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F 'indicator.example' "$COURSE_DATA/ioc.log" > "$COURSE_OUT/literal.txt"
grep 'indicator.example' "$COURSE_DATA/ioc.log" > "$COURSE_OUT/regex.txt"
test "$(wc -l < "$COURSE_OUT/literal.txt")" -eq 1
test "$(wc -l < "$COURSE_OUT/regex.txt")" -eq 2
printf 'literal=1 regex=2\n'


### 2. 부분 문자열과 필드의 정확한 값 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F '192.0.2.10' "$COURSE_DATA/ioc.log" > "$COURSE_OUT/substring.txt"
awk '$2 == "src=192.0.2.10" {print}' "$COURSE_DATA/ioc.log" > "$COURSE_OUT/exact-field.txt"
test "$(wc -l < "$COURSE_OUT/substring.txt")" -eq 2
test "$(wc -l < "$COURSE_OUT/exact-field.txt")" -eq 1
printf 'substring=2 exact_field=1\n'


### 3. 검색 결과 없음과 오류 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
status=0
grep -F -- 'absent-marker' "$COURSE_DATA/ioc.log" > "$COURSE_OUT/no-match.txt" || status=$?
test "$status" -eq 1
test ! -s "$COURSE_OUT/no-match.txt"
printf 'no_match_status=1\n'


### 4. 반복 입력을 코드로 실행하지 않기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
printf '%s\n' 'indicator.example' 'absent-marker' > "$COURSE_OUT/indicators.txt"
while IFS= read -r indicator; do
    status=0
    grep -F -- "$indicator" "$COURSE_DATA/ioc.log" > /dev/null || status=$?
    case $status in
        0) printf 'found=%s\n' "$indicator" ;;
        1) printf 'not_found=%s\n' "$indicator" ;;
        *) exit "$status" ;;
    esac
done < "$COURSE_OUT/indicators.txt"


### 5. getopts로 입력을 받는 검색 도구 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
bash "$COURSE_TOOLS/ioc-search.sh" -f "$COURSE_DATA/ioc.log" -i 'indicator.example' > "$COURSE_OUT/cli.txt"
test "$(wc -l < "$COURSE_OUT/cli.txt")" -eq 1
grep -q '^3:' "$COURSE_OUT/cli.txt"
printf 'cli_literal_match=1\n'


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
